# Gold Layer — Student Outcomes & Institutional KPIs
Produces 6 gold tables: cohort outcomes, course performance, retention analysis, faculty workload, weekly trends, and student scorecards.

In [ ]:
from pyspark.sql.functions import (
    col, count, sum as spark_sum, avg, round as spark_round,
    countDistinct, when, lit, weekofyear, year, max as spark_max,
    min as spark_min, current_timestamp
)
import pyspark.sql.functions as F

df_faculty     = spark.read.format('delta').table('silver_faculty')
df_students    = spark.read.format('delta').table('silver_students')
df_enrolments  = spark.read.format('delta').table('silver_enrolments')
df_assessments = spark.read.format('delta').table('silver_assessments')

In [ ]:
# Gold 1 — Cohort outcomes (by cohort_year and programme)
cohort_outcomes = (
    df_students
    .join(
        df_assessments.groupBy('student_id').agg(
            spark_round(avg('score'), 2).alias('avg_score'),
            spark_round(avg('is_pass') * 100, 2).alias('pass_rate'),
            count('assessment_id').alias('total_assessments'),
            spark_sum('is_resit').alias('resit_count'),
        ),
        'student_id', 'left'
    )
    .groupBy('cohort_year', 'programme', 'department', 'level')
    .agg(
        count('student_id').alias('total_students'),
        spark_sum('is_active').alias('active_students'),
        spark_sum('is_withdrawn').alias('withdrawn_students'),
        spark_round(avg('avg_score'), 2).alias('cohort_avg_score'),
        spark_round(avg('pass_rate'), 2).alias('cohort_pass_rate'),
    )
    .withColumn('retention_rate',
        spark_round(
            (col('total_students') - col('withdrawn_students')) / col('total_students') * 100, 2
        )
    )
    .withColumn('gold_timestamp', current_timestamp())
)
cohort_outcomes.write.mode('overwrite').format('delta').saveAsTable('gold_cohort_outcomes')
print(f'Gold cohort outcomes: {spark.table("gold_cohort_outcomes").count()} rows')

In [ ]:
# Gold 2 — Course performance
# `level` is a property of the enrolment (UG/PG/PhD), not of the assessment table,
# so bring it in from silver_enrolments via enrolment_id before grouping.
course_perf = (
    df_assessments
    .join(df_enrolments.select('enrolment_id', 'level'), 'enrolment_id', 'left')
    .groupBy('course_id', 'department', 'level', 'assessment_type')
    .agg(
        count('assessment_id').alias('total_submissions'),
        countDistinct('student_id').alias('unique_students'),
        spark_round(avg('score'), 2).alias('avg_score'),
        spark_round(avg('is_pass') * 100, 2).alias('pass_rate'),
        spark_sum('is_resit').alias('resit_count'),
        spark_round(avg('attempt_number'), 2).alias('avg_attempts'),
    )
    .withColumn('resit_rate',
        spark_round(col('resit_count') / col('total_submissions') * 100, 2)
    )
    .withColumn('gold_timestamp', current_timestamp())
)
course_perf.write.mode('overwrite').format('delta').saveAsTable('gold_course_performance')
print(f'Gold course performance: {spark.table("gold_course_performance").count()} rows')


In [ ]:
# Gold 3 — Retention analysis (at-risk students: withdrawn or low pass rate)
student_scores = (
    df_assessments
    .groupBy('student_id')
    .agg(
        spark_round(avg('score'), 2).alias('avg_score'),
        spark_round(avg('is_pass') * 100, 2).alias('pass_rate'),
        count('assessment_id').alias('total_assessments'),
        spark_sum('is_resit').alias('resit_count'),
    )
)
retention_analysis = (
    df_students
    .join(student_scores, 'student_id', 'left')
    .join(
        df_enrolments.groupBy('student_id').agg(
            count('enrolment_id').alias('total_enrolments'),
            spark_sum('is_completed').alias('completed_enrolments'),
            spark_sum('is_withdrawn').alias('withdrawn_enrolments'),
        ),
        'student_id', 'left'
    )
    .withColumn('retention_risk',
        when((col('pass_rate') < 40) | (col('is_withdrawn') == 1), 'High')
        .when(col('pass_rate') < 55, 'Medium')
        .otherwise('Low')
    )
    .withColumn('gold_timestamp', current_timestamp())
)
retention_analysis.write.mode('overwrite').format('delta').saveAsTable('gold_retention_analysis')
print(f'Gold retention analysis: {spark.table("gold_retention_analysis").count()} rows')

In [ ]:
# Gold 4 — Faculty workload
faculty_workload = (
    df_faculty
    .join(
        df_enrolments.groupBy('department').agg(
            count('enrolment_id').alias('dept_total_enrolments'),
            countDistinct('student_id').alias('dept_unique_students'),
        ),
        'department', 'left'
    )
    .withColumn('students_per_course',
        spark_round(col('dept_unique_students') / col('courses_assigned'), 1)
    )
    .withColumn('gold_timestamp', current_timestamp())
)
faculty_workload.write.mode('overwrite').format('delta').saveAsTable('gold_faculty_workload')
print(f'Gold faculty workload: {spark.table("gold_faculty_workload").count()} rows')

In [ ]:
# Gold 5 — Weekly submission trends
weekly_trends = (
    df_assessments
    .withColumn('week', weekofyear('submitted_date'))
    .withColumn('yr',   year('submitted_date'))
    .groupBy('yr', 'week', 'department')
    .agg(
        count('assessment_id').alias('submissions'),
        spark_round(avg('score'), 2).alias('avg_score'),
        spark_round(avg('is_pass') * 100, 2).alias('pass_rate'),
        spark_sum('is_resit').alias('resits'),
    )
    .withColumn('gold_timestamp', current_timestamp())
)
weekly_trends.write.mode('overwrite').format('delta').saveAsTable('gold_weekly_submission_trends')
print(f'Gold weekly submission trends: {spark.table("gold_weekly_submission_trends").count()} rows')

In [ ]:
# Gold 6 — Student scorecards (all KPIs per student)
student_scorecards = (
    retention_analysis
    .withColumn('performance_band',
        when(col('avg_score') >= 70, 'Distinction')
        .when(col('avg_score') >= 60, 'Merit')
        .when(col('avg_score') >= 50, 'Pass')
        .when(col('avg_score') >= 40, 'Near Pass')
        .otherwise('Fail')
    )
)
student_scorecards.write.mode('overwrite').format('delta').saveAsTable('gold_student_scorecards')
print(f'Gold student scorecards: {spark.table("gold_student_scorecards").count()} rows')

In [ ]:
print('Gold aggregation complete.')
print(f'  Cohort outcomes         : {cohort_outcomes.count():>6,}')
print(f'  Course performance      : {course_perf.count():>6,}')
print(f'  Retention analysis      : {retention_analysis.count():>6,}')
print(f'  Faculty workload        : {faculty_workload.count():>6,}')
print(f'  Weekly submission trends: {weekly_trends.count():>6,}')
print(f'  Student scorecards      : {student_scorecards.count():>6,}')